# LeetCode 3640 — Trionic Array II bằng Python

- Link: https://leetcode.com/problems/trionic-array-ii/
- Độ khó: **Hard**
- Chủ đề: **DP máy trạng thái** trên mảng con liên tiếp (tăng → giảm → tăng), tổng lớn nhất kiểu Kadane, phân đoạn đơn điệu

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxSumTrionic(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Mô tả "mảng con trionic" bằng **3 pha**: tăng ngặt (≥ 2 phần tử), giảm ngặt (≥ 1 bước), tăng ngặt (≥ 1 bước).
2. Thiết kế DP `inc1[i]`, `dec[i]`, `inc2[i]` = tổng lớn nhất của mảng con **kết thúc tại `i`** đang ở pha tương ứng (đã đủ điều kiện độ dài tối thiểu).
3. Hiểu vì sao lấy `max` giữa "kéo dài pha cũ" và "chuyển từ pha trước" — giống Kadane nhưng có ràng buộc hình dạng.
4. So sánh với cách phân đoạn đơn điệu (doocs) và với vét cạn.
5. Xử lý số âm (đáp án có thể âm).

## 2. Đề bài & ví dụ

Mảng con `nums[l..r]` là **trionic** nếu có `l < p < q < r` với `nums[l..p]` tăng ngặt, `nums[p..q]` giảm ngặt, `nums[q..r]` tăng ngặt. Tìm tổng lớn nhất của một mảng con trionic (đề bảo đảm tồn tại).

**Ví dụ 1:** `[0,-2,-1,-3,0,2,-1]` → `-4` (`[-2,-1,-3,0,2]`).

**Ví dụ 2:** `[1,4,2,7]` → `14`.

**Ràng buộc:** `4 ≤ n ≤ 10^5`, `|nums[i]| ≤ 10^9`.

In [1]:
from typing import List
import random, time

NEG = float("-inf")
examples = [([0, -2, -1, -3, 0, 2, -1], -4), ([1, 4, 2, 7], 14)]
for a, exp in examples:
    print(a, "-> kỳ vọng", exp)

[0, -2, -1, -3, 0, 2, -1] -> kỳ vọng -4
[1, 4, 2, 7] -> kỳ vọng 14


## 3. Kiến thức nền

### 3.1. Kiểm tra hình dạng trionic

Duyệt các cặp kề: chuỗi dấu `+` (tăng), `−` (giảm) phải có dạng `+^a −^b +^c` với `a, b, c ≥ 1` và không có cặp bằng nhau.

In [2]:
def is_trionic(seg):
    signs = []
    for x, y in zip(seg, seg[1:]):
        if x == y: return False
        signs.append("+" if y > x else "-")
    s = "".join(signs)
    import re
    return re.fullmatch(r"\++-+\++", s) is not None

print(is_trionic([1, 4, 2, 7]), is_trionic([1, 4, 2]), is_trionic([1, 2, 3, 2, 1, 5]), is_trionic([1, 4, 4, 2, 7]))

True False True False


### 3.2. Kadane có ràng buộc

Kadane: `best_end[i] = max(best_end[i−1], 0) + a[i]`. Ở đây mỗi pha chỉ được "kéo dài" khi cặp kề đúng chiều, và chỉ được "bắt đầu" từ pha trước (hoặc từ phần tử `i−1` với pha đầu).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với `i ≥ 1`:

- Nếu `a[i−1] < a[i]` (bước tăng):
  - `inc1[i] = max(inc1[i−1], a[i−1]) + a[i]` — kéo dài pha 1 hoặc bắt đầu pha 1 tại `i−1`.
  - `inc2[i] = max(inc2[i−1], dec[i−1]) + a[i]` — kéo dài pha 3 hoặc chuyển từ pha 2.
- Nếu `a[i−1] > a[i]` (bước giảm): `dec[i] = max(dec[i−1], inc1[i−1]) + a[i]`.
- Bước nào không hợp lệ cho pha nào thì pha đó `= −∞` tại `i`.

Đáp án `max_i inc2[i]`.

Ví dụ 2 `[1,4,2,7]`: `inc1[1] = 1 + 4 = 5`; `dec[2] = 5 + 2 = 7`; `inc2[3] = 7 + 7 = 14`. ✔

## 5. Cách 1 — Brute Force: mọi mảng con

In [3]:
def brute(nums):
    n = len(nums); best = NEG
    for l in range(n):
        for r in range(l + 3, n):
            seg = nums[l:r + 1]
            if is_trionic(seg): best = max(best, sum(seg))
    return best

for a, exp in examples:
    print(brute(a), "kỳ vọng", exp)

-4 kỳ vọng -4
14 kỳ vọng 14


**Nút thắt:** `O(n³)`.

## 6. Ý tưởng thuật toán / bước đột phá

Mỗi mảng con trionic kết thúc tại `r` được xác định bởi các điểm `l, p, q`. Thay vì liệt kê, DP giữ **tổng tốt nhất** cho mỗi pha kết thúc tại vị trí hiện tại. Vì các pha phải liên tiếp và mỗi phần tử chỉ thuộc một pha (trừ các điểm `p`, `q` là chỗ chuyển — được gán cho pha sau trong công thức), mỗi bước chỉ cần các giá trị tại `i−1` ⇒ `O(n)` thời gian, `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

- **Cách 2:** `O(n²)` — cố định `l`, quét sang phải với máy trạng thái, cộng dồn tổng.
- **Cách 3:** phân đoạn đơn điệu (doocs): tìm các cụm `tăng–giảm–tăng`, mở rộng hai đầu theo tổng lớn nhất.
- **Cách 4:** DP máy trạng thái `O(n)` (lời giải nộp).

### 7.1. Cách 2 — Cố định `l`

In [4]:
def fixed_left(nums):
    n = len(nums); best = NEG
    for l in range(n):
        phase = 0; s = nums[l]; steps = [0, 0, 0]
        for r in range(l + 1, n):
            if nums[r] == nums[r - 1]: break
            up = nums[r] > nums[r - 1]
            if phase == 0:
                if up: steps[0] += 1
                elif steps[0] >= 1: phase = 1; steps[1] += 1
                else: break
            elif phase == 1:
                if not up: steps[1] += 1
                else: phase = 2; steps[2] += 1
            else:
                if up: steps[2] += 1
                else: break
            s += nums[r]
            if phase == 2: best = max(best, s)
    return best

for a, exp in examples:
    print(fixed_left(a), "kỳ vọng", exp)

-4 kỳ vọng -4
14 kỳ vọng 14


### 7.2. Cách 3 — Phân đoạn đơn điệu

In [5]:
def segments(nums):
    n = len(nums); i = 0; ans = NEG
    while i < n:
        l = i; i += 1
        while i < n and nums[i - 1] < nums[i]: i += 1
        if i == l + 1: continue
        p = i - 1; s = nums[p - 1] + nums[p]
        while i < n and nums[i - 1] > nums[i]: s += nums[i]; i += 1
        if i == p + 1 or i == n or nums[i - 1] == nums[i]: continue
        q = i - 1; s += nums[i]; i += 1
        mx = t = 0
        while i < n and nums[i - 1] < nums[i]: t += nums[i]; i += 1; mx = max(mx, t)
        s += mx
        mx = t = 0
        for j in range(p - 2, l - 1, -1): t += nums[j]; mx = max(mx, t)
        s += mx
        ans = max(ans, s); i = q
    return ans

for a, exp in examples:
    print(segments(a), "kỳ vọng", exp)

-4 kỳ vọng -4
14 kỳ vọng 14


**Ý tưởng:** với mỗi đỉnh `p` (cuối một đoạn tăng) và đáy `q` kế tiếp, phần lõi bắt buộc là `nums[p−1..q+1]`; phần mở rộng trái (trong đoạn tăng) và phải (trong đoạn tăng sau) chọn **tiền tố/hậu tố có tổng lớn nhất** (có thể rỗng).

### 7.3. Cách 4 — DP máy trạng thái (lời giải chính)

In [6]:
def dp_states(nums):
    inc1 = dec = inc2 = NEG; best = NEG
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if a < b:
            n_inc1 = max(inc1, a) + b
            n_inc2 = max(inc2, dec) + b
            n_dec = NEG
        elif a > b:
            n_dec = max(dec, inc1) + b
            n_inc1 = n_inc2 = NEG
        else:
            n_inc1 = n_dec = n_inc2 = NEG
        inc1, dec, inc2 = n_inc1, n_dec, n_inc2
        best = max(best, inc2)
    return best

for a, exp in examples:
    print(dp_states(a), "kỳ vọng", exp)

-4 kỳ vọng -4
14 kỳ vọng 14


## 8. TRACE từng bước

In [7]:
def trace(nums):
    inc1 = dec = inc2 = NEG; best = NEG
    fmt = lambda v: "  -∞" if v == NEG else f"{v:>4}"
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if a < b: inc1, dec, inc2 = max(inc1, a) + b, NEG, max(inc2, dec) + b; kind = "tăng"
        elif a > b: inc1, dec, inc2 = NEG, max(dec, inc1) + b, NEG; kind = "giảm"
        else: inc1 = dec = inc2 = NEG; kind = "bằng"
        best = max(best, inc2)
        print(f"i={i} ({a}->{b}, {kind}): inc1={fmt(inc1)} dec={fmt(dec)} inc2={fmt(inc2)} | best={fmt(best)}")

trace([0, -2, -1, -3, 0, 2, -1])

i=1 (0->-2, giảm): inc1=  -∞ dec=  -∞ inc2=  -∞ | best=  -∞
i=2 (-2->-1, tăng): inc1=  -3 dec=  -∞ inc2=  -∞ | best=  -∞
i=3 (-1->-3, giảm): inc1=  -∞ dec=  -6 inc2=  -∞ | best=  -∞
i=4 (-3->0, tăng): inc1=  -3 dec=  -∞ inc2=  -6 | best=  -6
i=5 (0->2, tăng): inc1=   2 dec=  -∞ inc2=  -4 | best=  -4
i=6 (2->-1, giảm): inc1=  -∞ dec=   1 inc2=  -∞ | best=  -4


In [8]:
trace([5, 1, 3, 9, 4, 2, 6, 8, 1])
print("brute:", brute([5, 1, 3, 9, 4, 2, 6, 8, 1]))

i=1 (5->1, giảm): inc1=  -∞ dec=  -∞ inc2=  -∞ | best=  -∞
i=2 (1->3, tăng): inc1=   4 dec=  -∞ inc2=  -∞ | best=  -∞
i=3 (3->9, tăng): inc1=  13 dec=  -∞ inc2=  -∞ | best=  -∞
i=4 (9->4, giảm): inc1=  -∞ dec=  17 inc2=  -∞ | best=  -∞
i=5 (4->2, giảm): inc1=  -∞ dec=  19 inc2=  -∞ | best=  -∞
i=6 (2->6, tăng): inc1=   8 dec=  -∞ inc2=  25 | best=  25
i=7 (6->8, tăng): inc1=  16 dec=  -∞ inc2=  33 | best=  33
i=8 (8->1, giảm): inc1=  -∞ dec=  17 inc2=  -∞ | best=  33
brute: 33


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxSumTrionic(self, nums: List[int]) -> int:
        NEG = float('-inf')
        inc1 = dec = inc2 = NEG
        best = NEG
        for i in range(1, len(nums)):
            a, b = nums[i - 1], nums[i]
            if a < b:
                inc1, dec, inc2 = max(inc1, a) + b, NEG, max(inc2, dec) + b
            elif a > b:
                inc1, dec, inc2 = NEG, max(dec, inc1) + b, NEG
            else:
                inc1 = dec = inc2 = NEG
            if inc2 > best:
                best = inc2
        return best

sol = Solution()
for a, exp in examples:
    got = sol.maxSumTrionic(a)
    print(a, "->", got, "✔" if got == exp else "✘")

[0, -2, -1, -3, 0, 2, -1] -> -4 ✔
[1, 4, 2, 7] -> 14 ✔


## 10. Vì sao thuật toán đúng

**Định nghĩa trạng thái** (cho mảng con kết thúc tại `i`):

- `inc1[i]`: tổng lớn nhất của `nums[l..i]` tăng ngặt với `i − l ≥ 1` (pha 1 có ít nhất 2 phần tử, `i` có thể làm `p`).
- `dec[i]`: tổng lớn nhất của `nums[l..i]` có dạng `tăng (≥1 bước)` rồi `giảm (≥1 bước)`, kết thúc tại `i` (`i` có thể làm `q`).
- `inc2[i]`: tổng lớn nhất của mảng con trionic kết thúc tại `i`.

**Chuyển trạng thái:** mảng con thuộc `inc1[i]` với bước cuối tăng: hoặc là `[i−1, i]` (tổng `a + b`), hoặc là mảng thuộc `inc1[i−1]` kéo dài ⇒ `max(inc1[i−1], a) + b`. Mảng thuộc `dec[i]` (bước cuối giảm): bỏ `i` đi được mảng kết thúc tại `i−1` ở pha giảm (`dec[i−1]`) hoặc vừa hết pha 1 (`inc1[i−1]`, khi đó `i−1 = p`). Tương tự cho `inc2[i]`. Chiều ngược lại hiển nhiên. Lấy `max` tại mỗi `i` cho đáp án.

Bước "bằng nhau" phá mọi pha (đều đòi ngặt).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n³)` | `O(n)` |
| 2. Cố định `l` | `O(n²)` | `O(1)` |
| 3. Phân đoạn | `O(n)` (mỗi phần tử được duyệt hằng số lần) | `O(1)` |
| 4. DP máy trạng thái | `O(n)` | `O(1)` |

## 12. Bộ test

In [10]:
METHODS = [brute, fixed_left, segments, dp_states, lambda a: Solution().maxSumTrionic(a)]
for a, exp in examples:
    assert all(f(a) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().maxSumTrionic([1, 3, 2, 4]) == 10
assert Solution().maxSumTrionic([-5, -1, -9, -2]) == -17
assert Solution().maxSumTrionic([1, 2, 2, 3, 1, 4]) == brute([1, 2, 2, 3, 1, 4])        # có cặp bằng nhau
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên (chỉ giữ mảng có ít nhất một mảng con trionic)
rng = random.Random(3640)
cnt = 0
while cnt < 1500:
    a = [rng.randint(-6, 6) for _ in range(rng.randint(4, 12))]
    b = brute(a)
    if b == NEG: continue
    cnt += 1
    assert all(f(a) == b for f in METHODS[1:]), a
print("1500 test ngẫu nhiên: OK")

1500 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn
rng = random.Random(1)
a = [rng.randint(-10**9, 10**9) for _ in range(10**5)]
t0 = time.time(); x = Solution().maxSumTrionic(a); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 3), "s | khớp phân đoạn:", x == segments(a))

đáp án: 5804596876 | thời gian: 0.017 s | khớp phân đoạn: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép pha 1 chỉ có **một** phần tử (bắt đầu giảm ngay)

Đề đòi `l < p`: pha tăng đầu phải có ít nhất 2 phần tử.

In [14]:
def wrong_short_first(nums):
    inc1 = dec = inc2 = NEG; best = NEG
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if a < b: inc1, dec, inc2 = max(inc1, a) + b, NEG, max(inc2, dec) + b
        elif a > b: inc1, dec, inc2 = NEG, max(dec, inc1, a) + b, NEG                 # SAI: cho phép bắt đầu ở a
        else: inc1 = dec = inc2 = NEG
        best = max(best, inc2)
    return best

print("sai:", wrong_short_first([9, 1, 2, -5, -6, 1]), "| đúng:", brute([9, 1, 2, -5, -6, 1]))

sai: 12 | đúng: -7


### Lỗi 2 — Kéo dài pha bằng Kadane "bỏ phần âm" (cắt giữa pha)

Mảng con phải liên tiếp và giữ hình dạng; không được "khởi động lại" pha 2 hay pha 3 tại phần tử bất kỳ chỉ vì tổng âm.

In [15]:
def wrong_restart(nums):
    inc1 = dec = inc2 = NEG; best = NEG
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if a < b: inc1, dec, inc2 = max(inc1, a) + b, NEG, max(inc2, dec, 0) + b           # SAI: max với 0
        elif a > b: inc1, dec, inc2 = NEG, max(dec, inc1) + b, NEG
        else: inc1 = dec = inc2 = NEG
        best = max(best, inc2)
    return best

rng = random.Random(7)
while True:
    a = [rng.randint(-6, 6) for _ in range(rng.randint(4, 9))]
    if brute(a) != NEG and wrong_restart(a) != brute(a): break
print("phản ví dụ:", a, "| sai:", wrong_restart(a), "| đúng:", brute(a))

phản ví dụ: [-4, 0, 4, -6, -5, 2] | sai: 4 | đúng: -5


### Lỗi 3 — Cập nhật tuần tự làm trạng thái mới dùng lẫn trạng thái cùng bước

Ví dụ cập nhật `inc1` trước rồi dùng `inc1` **mới** để tính `dec` trong cùng bước: không thể xảy ra ở đây vì `inc1` và `dec` không cùng được cập nhật một bước, nhưng `inc2` dùng `dec` cũ — nếu gán `dec = NEG` trước khi tính `inc2` sẽ mất chuyển trạng thái.

In [16]:
def wrong_order(nums):
    inc1 = dec = inc2 = NEG; best = NEG
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if a < b:
            inc1 = max(inc1, a) + b; dec = NEG; inc2 = max(inc2, dec) + b            # SAI: dec đã bị xoá
        elif a > b:
            dec = max(dec, inc1) + b; inc1 = NEG; inc2 = NEG
        else: inc1 = dec = inc2 = NEG
        best = max(best, inc2)
    return best

print("sai:", wrong_order([1, 4, 2, 7]), "| đúng:", brute([1, 4, 2, 7]))

sai: -inf | đúng: 14


### Lỗi 4 — Khởi tạo `best = 0`

Đáp án có thể âm (ví dụ 1 là `−4`).

In [17]:
def wrong_zero_init(nums):
    r = dp_states(nums); return max(r, 0)                                             # SAI

print("sai:", wrong_zero_init([0, -2, -1, -3, 0, 2, -1]), "| đúng:", brute([0, -2, -1, -3, 0, 2, -1]))

sai: 0 | đúng: -4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mảng con có **hình dạng gồm nhiều pha** ⇒ DP máy trạng thái "đang ở pha nào", kết thúc tại `i`.
- Mỗi pha có độ dài tối thiểu ⇒ mã hoá vào điều kiện "bắt đầu pha" (ví dụ pha 1 bắt đầu bằng cặp `a[i−1], a[i]`).
- Tổng lớn nhất ⇒ `max(kéo dài, chuyển pha) + a[i]` (Kadane có ràng buộc).
- Đáp án có thể âm ⇒ khởi tạo `−∞`.

## 15. Biến thể của bài

1. **Bản I**: chỉ kiểm tra cả mảng có phải trionic không.
2. **Đếm số mảng con trionic**: thay `max` bằng **đếm** (số cách) trong cùng máy trạng thái.
3. **`k` pha luân phiên**: trạng thái `0..k−1`.
4. **Độ dài lớn nhất** thay vì tổng lớn nhất: cộng 1 thay cho `a[i]`.

In [18]:
# Biến thể 2: đếm số mảng con trionic
def count_trionic(nums):
    inc1 = dec = inc2 = 0; total = 0
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if a < b: inc1, dec, inc2 = inc1 + 1, 0, inc2 + dec
        elif a > b: inc1, dec, inc2 = 0, dec + inc1, 0
        else: inc1 = dec = inc2 = 0
        total += inc2
    return total

rng = random.Random(15)
for _ in range(500):
    a = [rng.randint(-4, 4) for _ in range(rng.randint(4, 11))]
    exp = sum(1 for l in range(len(a)) for r in range(l + 3, len(a)) if is_trionic(a[l:r + 1]))
    assert count_trionic(a) == exp
print("Biến thể 2 OK:", count_trionic([1, 4, 2, 7]), count_trionic([1, 3, 2, 4, 3, 5]))

Biến thể 2 OK: 1 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Kiểm tra mọi mảng con | `O(n³)` | Kiểm chứng |
| 2. Cố định `l` | Quét phải | `O(n²)` | `n` vừa |
| 3. Phân đoạn | Lõi + mở rộng tốt nhất | `O(n)` | Khi quen phân tích hình dạng |
| 4. DP máy trạng thái | 3 pha | `O(n)` | **Lời giải chuẩn**, dễ tổng quát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maxSumTrionic([1,2,1,2])`.

**Bài 2 (tính tay).** Trace `inc1, dec, inc2` cho `[2,5,3,1,4,6]`.

**Bài 3 (điền TODO).** Viết `longest_trionic(nums)` — độ dài lớn nhất của mảng con trionic (0 nếu không có).

**Bài 4 (tự viết lại).** Viết lại lời giải trả về luôn `(l, r)` của mảng con tối ưu.

**Bài 5 (mini challenge).** Tổng quát: mảng con gồm `k` pha luân phiên bắt đầu bằng tăng, mỗi pha ≥ 1 bước (pha đầu ≥ 1 bước như các pha khác). Tìm tổng lớn nhất.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def longest_trionic(nums):
    # TODO
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_k_phases(nums, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ có mảng con `[1,2,1,2]`. (2) Tổng `6`. (3) `6`.

**Bài 2.** (1) `i=1` (2→5): `inc1 = 7`. (2) `i=2` (5→3): `dec = 10`; `i=3` (3→1): `dec = 11`. (3) `i=4` (1→4): `inc2 = 15`; `i=5`: `inc2 = 21` ⇒ đáp án `21`.

**Bài 3.** (1) Như DP nhưng cộng 1. (2) Bắt đầu pha 1 bằng 2. (3) Kiểm chứng vét cạn.

**Bài 4.** (1) Lưu điểm bắt đầu kèm mỗi trạng thái. (2) Khi `max` chọn nhánh nào, sao chép điểm bắt đầu nhánh đó. (3) Trả `(start, i)` của `best`.

**Bài 5.** (1) Mảng `st[0..k−1]`. (2) Pha chẵn tăng, lẻ giảm. (3) `st[j] = max(st[j], st[j−1]) + b` nếu bước đúng chiều pha `j`; `st[0]` bắt đầu từ `a`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().maxSumTrionic([1, 2, 1, 2]) == 6
assert Solution().maxSumTrionic([2, 5, 3, 1, 4, 6]) == 21 == brute([2, 5, 3, 1, 4, 6])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def longest_trionic_sol(nums):
    inc1 = dec = inc2 = NEG; best = 0
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        if a < b: inc1, dec, inc2 = max(inc1, 1) + 1, NEG, max(inc2, dec) + 1
        elif a > b: inc1, dec, inc2 = NEG, max(dec, inc1) + 1, NEG
        else: inc1 = dec = inc2 = NEG
        best = max(best, inc2 if inc2 != NEG else 0)
    return best

def best_range(nums):
    S = [(NEG, -1)] * 3; best = (NEG, -1, -1)            # (tổng, điểm bắt đầu) cho inc1, dec, inc2
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]
        i1, d, i2 = S
        if a < b: S = [max(i1, (a, i - 1)), (NEG, -1), max(i2, d)]; S = [(S[0][0] + b, S[0][1]), S[1], (S[2][0] + b, S[2][1])]
        elif a > b: m = max(d, i1); S = [(NEG, -1), (m[0] + b, m[1]), (NEG, -1)]
        else: S = [(NEG, -1)] * 3
        if S[2][0] > best[0]: best = (S[2][0], S[2][1], i)
    return best

rng = random.Random(4)
for _ in range(500):
    a = [rng.randint(-5, 5) for _ in range(rng.randint(4, 11))]
    L = max((r - l + 1 for l in range(len(a)) for r in range(l + 3, len(a)) if is_trionic(a[l:r + 1])), default=0)
    assert longest_trionic_sol(a) == L
    b = brute(a)
    if b != NEG:
        s, l, r = best_range(a); assert s == b and is_trionic(a[l:r + 1]) and sum(a[l:r + 1]) == b
print("Bài 3, 4 OK; ví dụ 1:", best_range([0, -2, -1, -3, 0, 2, -1]))

Bài 3, 4 OK; ví dụ 1: (-4, 1, 5)


In [23]:
# Bài 5
def max_k_phases_sol(nums, k):
    st = [NEG] * k; best = NEG
    for i in range(1, len(nums)):
        a, b = nums[i - 1], nums[i]; new = [NEG] * k
        for j in range(k):
            want_up = j % 2 == 0
            if (b > a) if want_up else (b < a):
                prev = max(st[j], a if j == 0 else st[j - 1])
                new[j] = prev + b
        st = new; best = max(best, st[k - 1])
    return best

def brute_k(nums, k):
    best = NEG
    for l in range(len(nums)):
        for r in range(l + 1, len(nums)):
            seg = nums[l:r + 1]; sg = []
            if any(x == y for x, y in zip(seg, seg[1:])): continue
            for x, y in zip(seg, seg[1:]):
                c = "+" if y > x else "-"
                if not sg or sg[-1] != c: sg.append(c)
            if len(sg) == k and sg[0] == "+": best = max(best, sum(seg))
    return best

rng = random.Random(5)
for _ in range(400):
    a = [rng.randint(-5, 5) for _ in range(rng.randint(2, 10))]; k = rng.randint(1, 4)
    assert max_k_phases_sol(a, k) == brute_k(a, k), (a, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
inc1 = dec = inc2 = -inf; best = -inf
for i in 1..n-1:
    a, b = nums[i-1], nums[i]
    if a < b:   inc1, dec, inc2 = max(inc1, a) + b, -inf, max(inc2, dec) + b
    elif a > b: inc1, dec, inc2 = -inf, max(dec, inc1) + b, -inf
    else:       inc1 = dec = inc2 = -inf
    best = max(best, inc2)
```

**Checklist:**

- [ ] Pha 1 có ít nhất 2 phần tử (bắt đầu bằng `a`)?
- [ ] Bước bằng nhau xoá mọi pha?
- [ ] Gán đồng thời (dùng giá trị cũ)?
- [ ] `best` khởi tạo `−∞`?
- [ ] Đã so với vét cạn?